# Sistema de Análisis de Hojas de Vida
## NLP + BERT + Transformers

In [28]:
# Importar módulos
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import os
from pathlib import Path

## 1. Cargar Dataset

In [29]:
from src.data_processing.data_loader import DatasetLoader

loader = DatasetLoader()
datasets = loader.load_all()
print("Datasets cargados:", list(datasets.keys()))

Loading datasets...
Loading 1_resume_classification...
  - training_data.csv: 10000 rows, 7 columns
Loaded 2 datasets
Datasets cargados: ['resume_class_training', 'job_role_category_map']


## 2. Preparar Datos

In [30]:
from src.data_processing.preprocessor import DatasetPreprocessor

df = datasets['resume_class_training']
preprocessor = DatasetPreprocessor()
texts, labels = preprocessor.prepare_classification_data(df, "Resume Text", "Job Role")
train_texts, val_texts, train_labels, val_labels = preprocessor.prepare_for_bert(texts, labels, val_size=0.2)
label_mapping, reverse_mapping = preprocessor.get_label_mapping()
num_classes = len(label_mapping)
print(f"Train: {len(train_texts)}, Val: {len(val_texts)}")
print(f"Job Roles: {num_classes}")

Preparing classification data...
After dropping nulls/empty: 10000 samples
After cleaning: 10000 samples
After dedup: 10000 samples
Encoded 324 unique labels
Train: 8000, Validation: 2000
Train: 8000, Val: 2000
Job Roles: 324


## 3. Cargar o Entrenar Modelo

In [31]:
from src.ml_models.bert_classifier import BertClassifierModel

MODEL_PATH = '../models/bert_classifier.pt'

classifier = BertClassifierModel(num_classes=num_classes, device='cuda')

if os.path.exists(MODEL_PATH):
    print("✅ Cargando modelo guardado...")
    classifier.load_model(MODEL_PATH)
else:
    print("⚠️ Entrenando modelo por primera vez...")
    classifier.train(
        train_texts, train_labels,
        val_texts, val_labels,
        epochs=10,
        batch_size=32,
        save_path=MODEL_PATH
    )
    print("✅ Modelo guardado!")

[BERT] Dispositivo: cuda
[BERT] GPU: NVIDIA GeForce GTX 1070


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Cargando modelo guardado...


## 4. Predicción con Porcentajes

In [ ]:
sample_resume = """
Julieth Shirley

I am a respectful and honest person who constantly strives to achieve my goals, learn, and improve every day. Committed to creating a positive work environment, I am willing to perform various functions to contribute to the improvement of the company, as well as participate in training and educational programs for my professional growth.

As an administrative manager and administrative assistant, I possess solid knowledge in information technology, document production, customer service, document organization, organizational improvement programs, and activity coordination. I value the opportunity to take on new responsibilities, as I see them as a chance for both personal and professional growth and development.

PERSONAL INFORMATION
ID Number: 1'076.502.009
Date of Birth: June 1, 2005
Home Address: Calle 28 N° 51 Bis 10
City of Residence: Neiva, Huila
Phone: 3138597091 - 3117583067
Email: Juliethcruz0123@gmail.com / Juliethcruz0123@icloud.com
ADMINISTRATIVE MANAGER
PERSONAL REFERENCES
Alexandra Yamile Posada Arango
Position: Merchant
Phone: 3117583067
John Fredy Posada Arango
Position: Public Accountant
Phone: 3015669311
SKILLS
Teamwork
Information Technology
Assertive Communication
Customer Service
Time Management
Initiative and Proactivity
EDUCATION
Technologist in Administrative Management

SENA – Center for Industry, Business, and Services
Dec 2021 – Dec 2023

Technical Program in Administrative Assistance

SENA – Center for Industry, Business, and Services
Feb 2020 – Nov 2021

Human Resources Administration – Short Course

SENA – Agricultural Sector Service Center
40 hours – Aug 2022

Customer Service through Technological Media – Short Course

SENA – Agro-Business Center
48 hours – Sep 2022

English Intensive Program

Humberto Tafur Charry
Feb 2022 – Nov 2021

High School Diploma

Humberto Tafur Charry
Nov 2021

Mechatronics Engineering

Corporación Universitaria del Huila - Corhuila
Currently Studying

Law

Universidad Surcolombiana
Currently Studying

SOFTWARE
Microsoft Word
Microsoft Excel
Adobe
PowerPoint
WORK EXPERIENCE
Administrative Assistant

Inversiones Lion Hill S.A.S
Jun 2023 – Dec 2023
Phone: 3175853424

Organized accounting archives (expense and income invoices)
Answered phone calls and assisted customers
Performed activities related to the position both inside and outside the institution
Cashier and Sales Assistant

Tienda la Paisa
Jan 2021 – Jun 2023
Phone: 3117583067

Operated the cash register
Assisted customers
Organized products on shelves
Responded to customer inquiries
Maintained an organized and pleasant shopping environment focused on customer service
Administrative Assistant

Institución Educativa Enrique Olaya Herrera
Apr 2021 – Oct 2021
Phone: 3158828576

Organized and classified files according to regulations
Managed inventories using technical procedures
Assisted in projects following methodologies and regulations
Coordinated administrative meetings according to organizational procedures
Prepared documents following technical guidelines
Assisted customers according to service protocols
Waitress

Discoteca Ole
Dec 2023 – Present
Phone: 3194241352

Handled cash and processed credit card payments
Took orders and served drinks efficiently
Responded to customer requests professionally and courteously
Maintained tables clean and organized
WORK REFERENCES
Alexandra Yamile Posada Arango
Position: Merchant
Phone: 3117583067
Yeni Paola García Sarmiento
Position: Academic Secretary
Phone: 3053271510
"""

print("CV:", sample_resume[:200], "...")

CV: 
Julieth Shirley Soy una persona respetuosa y honesta que se esfuerza constantemente para alcanzar sus metas, aprender y mejorar cada día. Comprometida con la creación de un buen ambiente laboral, est ...


In [33]:
from src.text_preprocessing.text_cleaner import TextCleaner
import torch

cleaner = TextCleaner()
cleaned_text = cleaner.clean(sample_resume)
print("Texto limpio:", cleaned_text[:300], "...")

Texto limpio: julieth shirley soy una persona respetuosa y honesta que se esfuerza constantemente para alcanzar sus metas aprender y mejorar cada dia comprometida con la creacion de un buen ambiente laboral estoy dispuesta a realizar diversas funciones para contribuir al mejoramiento de la empresa asi como a part ...


In [34]:
# Predicción con probabilidades
def predict_with_probabilities(classifier, text, reverse_mapping, max_len=512):
    classifier.model.eval()
    encoding = classifier.tokenizer(
        text,
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )
    input_ids = encoding['input_ids'].to(classifier.device)
    attention_mask = encoding['attention_mask'].to(classifier.device)
    
    with torch.no_grad():
        outputs = classifier.model(input_ids, attention_mask)
        probs = torch.softmax(outputs, dim=1)
    
    probs = probs[0].cpu().numpy()
    
    results = []
    for idx, prob in enumerate(probs):
        job_role = reverse_mapping.get(idx, f"Unknown_{idx}")
        results.append({'job_role': job_role, 'probability': prob * 100})
    
    results.sort(key=lambda x: x['probability'], reverse=True)
    return results

predictions = predict_with_probabilities(classifier, cleaned_text, reverse_mapping)

print("=" * 60)
print("PREDICCIÓN DE JOB ROLE")
print("=" * 60)
print(f"\n🎯 Job Role: {predictions[0]['job_role']}")
print(f"📊 Confianza: {predictions[0]['probability']:.2f}%")
print("\nTOP 10 JOB ROLES:")
print("-" * 60)
for i, pred in enumerate(predictions[:10]):
    bar = "█" * int(pred['probability'] / 2)
    print(f"{i+1:2}. {pred['job_role']:35} {pred['probability']:6.2f}% {bar}")

PREDICCIÓN DE JOB ROLE

🎯 Job Role: Jupyter Developer
📊 Confianza: 9.84%

TOP 10 JOB ROLES:
------------------------------------------------------------
 1. Jupyter Developer                     9.84% ████
 2. Clinical Lab Scientist                3.16% █
 3. Kotlin Developer                      2.54% █
 4. Java Backend Developer                2.21% █
 5. Mobile Developer Android              1.45% 
 6. Technical Writer                      1.34% 
 7. Truck Driver                          1.31% 
 8. Environmental Engineer                1.29% 
 9. Drupal Developer                      1.26% 
10. Elixir Developer                      1.19% 


## 5. Extracción de Skills

In [35]:
from src.text_preprocessing.skill_extractor import SkillExtractor

skills_list = loader.get_skills_list()
skill_extractor = SkillExtractor(skills_list)
extracted_skills = skill_extractor.extract_from_resume(sample_resume)

print("🛠️ Skills detectados:")
for skill in extracted_skills:
    print(f"  ✓ {skill}")

🛠️ Skills detectados:
  ✓ ai
  ✓ go


In [36]:
print("\n" + "=" * 60)
print("RESUMEN FINAL")
print("=" * 60)
print(f"""
📄 CV ANALIZADO
   Caracteres: {len(cleaned_text)}

🎯 PREDICCIÓN
   Job Role: {predictions[0]['job_role']}
   Confianza: {predictions[0]['probability']:.2f}%
   Alternativas: {predictions[1]['job_role']} ({predictions[1]['probability']:.2f}%), {predictions[2]['job_role']} ({predictions[2]['probability']:.2f}%)

🛠️ SKILLS: {len(extracted_skills)} detectados

📊 Modelo: {num_classes} Job Roles
""")


RESUMEN FINAL

📄 CV ANALIZADO
   Caracteres: 3545

🎯 PREDICCIÓN
   Job Role: Jupyter Developer
   Confianza: 9.84%
   Alternativas: Clinical Lab Scientist (3.16%), Kotlin Developer (2.54%)

🛠️ SKILLS: 2 detectados

📊 Modelo: 324 Job Roles



---
## Notas:
- Si el modelo existe, lo carga automáticamente
- Si no existe, entrena y guarda
- El modelo se guarda en: ../models/bert_classifier.pt